# ساخت برنامه‌های تولید تصویر (OpenAI)

قابلیت‌های مدلسازی زبانی بزرگ فقط به تولید متن محدود نمی‌شود. شما همچنین می‌توانید تصاویر را از شرح‌های متنی تولید کنید. تصاویر به عنوان یک مدالیته در حوزه‌هایی مانند فناوری پزشکی، معماری، گردشگری، توسعه بازی، بازاریابی و غیره بسیار کاربردی هستند. در این درس با مدل‌های **GPT Image** کنونی در پلتفرم OpenAI کار می‌کنیم.

## اهداف یادگیری

تا پایان این درس شما قادر خواهید بود:

- توضیح دهید تولید تصویر چیست و در کجاها کاربرد دارد.
- خانواده مدل‌های `gpt-image` را بفهمید و تفاوت آن‌ها با مدل‌های قدیمی DALL·E را درک کنید.
- یک برنامه تولید تصویر بسازید و تصاویر را ویرایش کنید.

## تولید تصویر چیست؟

مدل‌های تولید تصویر، تصاویر را از یک ورودی متنی ایجاد می‌کنند. مدل‌های مدرن مانند `gpt-image` در طول آموزش رابطه بین متن و تصویر را یاد می‌گیرند و سپس به صورت تکراری نویز تصادفی را به تصویری تبدیل می‌کنند که با ورودی متنی شما مطابقت دارد.

دو خانواده شناخته شده مدل‌های تصویر به شرح زیرند:

- **`gpt-image` (OpenAI)** — نسل فعلی که در این درس استفاده می‌شود. از تولید تصویر از متن و ویرایش تصویر (نقاشی درون عکس با ماسک) پشتیبانی می‌کند.
- **Midjourney** — یک مدل محبوب شخص ثالث با سرویس اختصاصی و فرآیند کاری مبتنی بر Discord.

> مدل‌های قدیمی‌تر تصویر OpenAI — **DALL·E 2** و **DALL·E 3** — مدل‌های میراثی هستند. DALL·E 3 دیگر برای استقرارهای جدید در دسترس نیست، و قابلیت‌هایی مانند `create_variation` فقط در DALL·E 2 وجود داشت. برای برنامه‌های جدید از مدل‌های `gpt-image` استفاده کنید.

> **مهم:** مدل‌های `gpt-image` تصویر تولید شده را به صورت **base64** (`b64_json`) برمی‌گردانند، نه به صورت یک URL. کد شما رشته base64 را رمزگشایی می‌کند به بایت‌ها و ذخیره می‌کند — URL تصویری برای دانلود وجود ندارد.


## ساخت اولین برنامه تولید تصویر شما

پس ساخت یک برنامه تولید تصویر چه چیزهایی نیاز دارد؟ شما به کتابخانه‌های زیر نیاز دارید:

- **python-dotenv**، به شدت توصیه می‌شود از این کتابخانه برای نگهداری اسرار خود در یک فایل *.env* جدا از کد استفاده کنید.
- **openai**، این کتابخانه برای تعامل با API اِپن‌ای‌آی استفاده می‌شود.
- **pillow**، برای کار با تصاویر در پایتون.


1. یک فایل *.env* با محتوای زیر ایجاد کنید:

    ```text
    OPENAI_API_KEY='<add your OpenAI key here>'
    ```



1. کتابخانه‌های بالا را در فایلی به نام *requirements.txt* به این شکل جمع‌آوری کنید:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. سپس، محیط مجازی ایجاد کرده و کتابخانه‌ها را نصب کنید:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> برای ویندوز، از دستورات زیر برای ایجاد و فعال‌سازی محیط مجازی خود استفاده کنید:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. کد زیر را در فایلی به نام *app.py* اضافه کنید:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai

    # وارد کردن dotenv
    dotenv.load_dotenv()

    # ایجاد شیء OpenAI (خواندن OPENAI_API_KEY از فایل .env شما)
    client = openai.OpenAI()


    try:
        # ایجاد تصویر با استفاده از API تولید تصویر
        generation_response = client.images.generate(
            model="gpt-image-1",
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # متن درخواست خود را اینجا وارد کنید
            size='1024x1024',
            n=1
        )
        # تنظیم دایرکتوری برای ذخیره تصویر
        image_dir = os.path.join(os.curdir, 'images')

        # اگر دایرکتوری وجود نداشت، آن را ایجاد کنید
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # مقداردهی اولیه مسیر تصویر (توجه داشته باشید نوع فایل باید png باشد)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # مدل‌های gpt-image تصویر را به صورت base64 (b64_json) برمی‌گردانند، نه URL
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # نمایش تصویر در نمایشگر پیش‌فرض تصویر
        image = Image.open(image_path)
        image.show()

    # گرفتن استثناء‌ها
    except openai.BadRequestError as err:
        print(err)

    ```

بیایید این کد را توضیح دهیم:

- ابتدا کتابخانه‌هایی که نیاز داریم، از جمله کتابخانه OpenAI، کتابخانه dotenv، ماژول base64، و کتابخانه Pillow را وارد می‌کنیم.

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai
    ```

- سپس کلاینت را ایجاد می‌کنیم، که کلید API را از فایل ``.env`` می‌خواند.

    ```python
    # ایجاد شیء OpenAI
    client = openai.OpenAI()
    ```

- بعد، تصویر را تولید می‌کنیم:

    ```python
    generation_response = client.images.generate(
        model="gpt-image-1",
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # متن درخواست خود را اینجا وارد کنید
        size='1024x1024',
        n=1
    )
    ```

    مدل‌های `gpt-image` تصویر را به صورت رشته‌ای **base64** در `data[0].b64_json` برگشت می‌دهند. ما آن را به بایت‌ها رمزگشایی می‌کنیم و در یک فایل می‌نویسیم — هیچ URL برای دانلود وجود ندارد.

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- در نهایت، تصویر را باز می‌کنیم و با استفاده از نمایشگر تصویر استاندارد آن را نمایش می‌دهیم:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### جزییات بیشتر درباره تولید تصویر

بیایید پارامترهای `images.generate` را بررسی کنیم:

- **model**، مدلی است که برای تصویر استفاده می‌شود (برای مثال `gpt-image-1`).
- **prompt**، توصیف متنی است که برای تولید تصویر استفاده می‌شود. اینجا عبارت "Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils" است.
- **size**، اندازه تصویر تولید شده است (برای مثال `1024x1024`، `1536x1024`، `1024x1536`، یا `"auto"`).
- **n**، تعداد تصاویر تولید شده است. در اینجا، یکی تولید می‌کنیم.

> مدل‌های تصویر پارامتری به نام `temperature` ندارند — این پارامتر مربوط به تولید متن است. برای تنوع بیشتر، دوباره API را فراخوانی کنید؛ برای کاهش تنوع، توصیفتان را دقیق‌تر بیان کنید.

## قابلیت‌های اضافی تولید تصویر

شما دیدید چگونه با چند خط پایتون یک تصویر تولید کنید. مدل‌های `gpt-image` همچنین می‌توانند یک تصویر موجود را **ویرایش** کنند. با ارائه یک تصویر، یک **ماسک** اختیاری (که ناحیه‌ای که باید تغییر کند را نشان می‌دهد)، و یک توصیف، می‌توانید بخشی از تصویر را تغییر دهید — برای مثال، به خرگوشمان کلاهی اضافه کنید.

```python
response = client.images.edit(
  model="gpt-image-1",
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# تغییرات همچنین به صورت base64 بازگردانده می‌شوند
edited_image = base64.b64decode(response.data[0].b64_json)
```

تصویر پایه فقط شامل خرگوش است؛ تصویر نهایی کلاه را روی خرگوش دارد.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**سلب مسئولیت**:
این سند با استفاده از سرویس ترجمه هوش مصنوعی [Co-op Translator](https://github.com/Azure/co-op-translator) ترجمه شده است. در حالی که ما در تلاش برای دقت هستیم، لطفاً توجه داشته باشید که ترجمه‌های خودکار ممکن است شامل خطاها یا نادرستی‌هایی باشند. سند اصلی به زبان مادری خود باید به عنوان منبع معتبر در نظر گرفته شود. برای اطلاعات حیاتی، ترجمه حرفه‌ای انسانی توصیه می‌شود. ما در قبال هرگونه سوء تفاهم یا برداشت نادرست ناشی از استفاده از این ترجمه مسئولیتی نداریم.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
